In [1]:
from langgraph.graph import StateGraph, START, END 
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama

In [2]:
model = ChatOllama(model = "phi4-mini:latest")

In [3]:
# We will tell the model to generate a joke and then tell to explain the joke
class JokeState(TypedDict):
    topic: str 
    joke: str 
    explaination: str 

In [4]:
def generateJoke(state: JokeState):
    prompt = f"Generate a joke on the given topic: {state['topic']}."
    response = model.invoke(prompt)
    return {
        "joke" : response.content
    }

In [5]:
def explainJoke(state: JokeState):
    prompt = f"Explain this joke: {state['joke']}"
    response = model.invoke(prompt)
    return {
        "explaination": response.content
    }

In [18]:
graph = StateGraph(JokeState)

graph.add_node("generateJoke" , generateJoke)
graph.add_node("explainJoke" , explainJoke)

graph.add_edge(START , "generateJoke")
graph.add_edge("generateJoke" , "explainJoke")
graph.add_edge("explainJoke" , END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)

In [24]:
config1 = {
    "configurable": {
        "thread_id": "3"
    }
}
res = workflow.invoke(
    {"topic": "Bengali"}, config = config1 
)

In [25]:
workflow.get_state(config = config1)

StateSnapshot(values={'topic': 'Bengali', 'joke': "Why don't Bengalis like to play hide and seek? Because good luck hiding when everyone knows you're 'Choleric'! 😄🧠😅", 'explaination': 'This is a light-hearted pun that relies on the wordplay surrounding "choleric" as it relates specifically within Bengali culture. In some contexts, being described (or perceived) as choleric can imply someone who easily gets angry or irritable—a trait humorously assumed by others when referring to Bengalis.\n\nThe joke explains why playing hide and seek might be challenging for certain groups: if everyone knows you have a tendency toward becoming irritated ("Choleric"), it becomes difficult not just physically but emotionally (or socially) because they expect that your mood will change quickly, potentially making the game less enjoyable or successful. The humor comes from using an idiom that\'s culturally specific and then relating back to hide and seek—a universal childhood activity—thereby creating a p

In [23]:
workflow.get_state(config = {"configurable": {"thread_id": "2"}})

StateSnapshot(values={'topic': 'Asian', 'joke': "Sure, here's one that's light-hearted and culturally sensitive:\n\nWhy don't you ever see an Asian elephant at parties? Because he's always in charge of doing karaoke! 😂\n\nRemember to share jokes with respect for all cultures; humor can vary widely depending on personal experiences or cultural backgrounds. Enjoy laughing responsibly! 🐘🎤✨", 'explaination': 'This joke involves a play on words and stereotypes that might not be appropriate due to its light-hearted take, even if it attempts to maintain sensitivity by involving elephants—a context commonly associated with Asian culture through the symbol of an elephant in festivals like Chinese New Year or Thai Songkran. The humor is derived from imagining that at parties where karaoke (a popular activity) happens regularly ("doing karaoke"), you\'d never see an "Asian" character because they\'re always busy singing.\n\nHowever, it\'s important to note how sensitive this topic can be and the 

In [27]:
# to get the intermediate state values
all_states = list(workflow.get_state_history(config = config1))

In [28]:
len(all_states)

4

In [ ]:
all_states[-1] # state before start node

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '3', 'checkpoint_ns': '', 'checkpoint_id': '1f1528d7-984e-65a5-bfff-f6d257cb0405'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-05-18T07:45:00.781507+00:00', parent_config=None, tasks=(PregelTask(id='cf0102a1-8ac7-4e6c-f5fc-4d1e5550b346', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'Bengali'}),), interrupts=())

In [30]:
all_states[-2] # state after start and before generation joke

StateSnapshot(values={'topic': 'Bengali'}, next=('generateJoke',), config={'configurable': {'thread_id': '3', 'checkpoint_ns': '', 'checkpoint_id': '1f1528d7-984e-65a6-8000-a03a34ff2910'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-18T07:45:00.781507+00:00', parent_config={'configurable': {'thread_id': '3', 'checkpoint_ns': '', 'checkpoint_id': '1f1528d7-984e-65a5-bfff-f6d257cb0405'}}, tasks=(PregelTask(id='0f89e4d8-4b7b-bd99-ad38-a788bb3983bf', name='generateJoke', path=('__pregel_pull', 'generateJoke'), error=None, interrupts=(), state=None, result={'joke': "Why don't Bengalis like to play hide and seek? Because good luck hiding when everyone knows you're 'Choleric'! 😄🧠😅"}),), interrupts=())